# Phase 1 — Data Preparation

### Purpose
Prepare clean and optimized datasets for downstream ML and API usage.

### Input
- data/raw/articles.csv
- data/raw/customers.csv
- data/raw/transactions_train.csv

### Output
- data/processed/transactions_small.csv
- data/processed/article_lookup.csv
- data/processed/customers_clean.csv

In [ ]:
import sys
import os

# This adds the parent directory (..) to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

import config

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import config

In [ ]:
articles = pd.read_csv(config.ARTICLES_RAW_PATH)

print("Shape:", articles.shape)
articles.head()

In [ ]:
articles.isna().sum().sort_values(ascending=False)

In [ ]:
total = len(articles)

null_count = articles["detail_desc"].isna().sum()
filled_count = total - null_count

null_percent = (null_count / total) * 100
filled_percent = (filled_count / total) * 100

print(f"Total rows: {total}")
print(f"Null values: {null_count} ({null_percent:.2f}%)")
print(f"Filled values: {filled_count} ({filled_percent:.2f}%)")

In [ ]:
articles["detail_desc"].dropna().sample(5)

In [ ]:
articles["detail_desc"] = articles["detail_desc"].fillna("")

In [ ]:
articles.groupby("product_group_name")["product_type_name"].nunique().sort_values(ascending=False)

In [ ]:
articles.groupby("product_type_name")["colour_group_name"].nunique().sort_values(ascending=False).head(10)

In [ ]:
articles.groupby("product_type_name")["graphical_appearance_name"].nunique().sort_values(ascending=False).head(10)

In [ ]:
articles["product_type_name"].value_counts().head(10)

In [ ]:
articles["product_group_name"].value_counts().head(10)

In [ ]:
articles["colour_group_name"].value_counts().head(10)

In [ ]:
articles.groupby(
    ["product_type_name", "colour_group_name"]
).size().sort_values(ascending=False).head(10)

In [ ]:
articles["style_key"] = (
    articles["product_type_name"] + " | " +
    articles["colour_group_name"] + " | " +
    articles["graphical_appearance_name"]
)

👉 Why useful:

Helps detect:
“Black T-shirt” → “Blue Jeans”
Adds fashion intelligence

In [ ]:
articles["product_family"] = articles["product_type_name"]

👉 Later you map:

Product	Complement
T-shirt	Jeans
Shirt	Trousers
Dress	Accessories

👉 This is used for:

Cross-selling (Amazon-style)

In [ ]:
articles["gender_category"] = articles["index_name"]

👉 Why:

Prevents wrong recommendations:
Men’s jeans → not shown to women

In [ ]:
article_lookup = articles[[
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "index_name",
    "style_key",
    "product_family",
    "gender_category"
]].copy()


In [ ]:
article_lookup["prod_name"] = article_lookup["prod_name"].fillna("Unknown Product")
article_lookup["product_type_name"] = article_lookup["product_type_name"].fillna("Unknown Type")
article_lookup["product_group_name"] = article_lookup["product_group_name"].fillna("Unknown Group")
article_lookup["graphical_appearance_name"] = article_lookup["graphical_appearance_name"].fillna("Unknown Appearance")
article_lookup["colour_group_name"] = article_lookup["colour_group_name"].fillna("Unknown Color")
article_lookup["index_name"] = article_lookup["index_name"].fillna("Unknown")

In [ ]:
article_lookup.to_csv(config.ARTICLE_LOOKUP_PATH, index=False)

In [ ]:
assert "article_id" in article_lookup.columns
assert article_lookup["article_id"].isna().sum() == 0
assert article_lookup.shape[0] > 100000


In [ ]:
article_lookup.head(10)

## CUSTOMERS DATA CLEANING ##


In [ ]:
customers = pd.read_csv(config.CUSTOMERS_RAW_PATH)


print("Original shape:", customers.shape)
print(customers.isna().sum())

In [ ]:
customers.head(10)

In [ ]:
total = len(customers)

cols = ["FN", "Active", "fashion_news_frequency"]

# Total nulls across selected columns
null_count = customers[cols].isna().sum().sum()

# Total possible values in these columns
total_values = total * len(cols)

filled_count = total_values - null_count

null_percent = (null_count / total_values) * 100
filled_percent = (filled_count / total_values) * 100

print(f"Total rows: {total}")
print(f"Total values checked: {total_values}")
print(f"Null values: {null_count} ({null_percent:.2f}%)")
print(f"Filled values: {filled_count} ({filled_percent:.2f}%)")
cols = ["FN", "Active", "fashion_news_frequency"]

for col in cols:
    nulls = customers[col].isna().sum()
    percent = (nulls / len(customers)) * 100

    print(f"{col}: {nulls} nulls ({percent:.2f}%)")

In [ ]:
customers = pd.read_csv(config.CUSTOMERS_RAW_PATH)

customers_clean = customers.copy()

In [ ]:
customers_clean["FN"] = customers_clean["FN"].fillna(0)
customers_clean["Active"] = customers_clean["Active"].fillna(0)

Filling with 0
Why:

You lose behavioral signals
These are cheap but useful features
Later useful for:
segmentation
churn prediction
marketing personalization

In [ ]:
# Missing indicators (important)
customers_clean["FN_missing"] = customers["FN"].isna().astype(int)
customers_clean["Active_missing"] = customers["Active"].isna().astype(int)

# Fashion news
customers_clean["fashion_news_frequency"] = customers_clean["fashion_news_frequency"].fillna("NONE")

In [ ]:
print(customers_clean[["FN", "Active", "fashion_news_frequency"]].isna().sum())

In [ ]:
print("FN:", customers_clean["FN"].unique())
print("Active:", customers_clean["Active"].unique())
print("Fashion News:", customers_clean["fashion_news_frequency"].unique())


In [ ]:
def age_group(age):
    if age < 25:
        return "Young"
    elif age < 40:
        return "Adult"
    elif age < 60:
        return "Middle"
    else:
        return "Senior"

customers_clean["age_group"] = customers_clean["age"].apply(age_group)

Creation of age group models does not need specific age they need group 

In [ ]:
def engagement_score(row):
    score = 0
    score += row["FN"]
    score += row["Active"]

    if row["fashion_news_frequency"] == "Regularly":
        score += 2
    elif row["fashion_news_frequency"] == "Monthly":
        score += 1

    return score

customers_clean["engagement_score"] = customers_clean.apply(engagement_score, axis=1)

Engaement score 

In [ ]:
customers_clean["club_member_status"] = customers_clean["club_member_status"].replace({
    "PRE-CREATE": "INACTIVE",
    "LEFT CLUB": "INACTIVE"
})

In [ ]:
customers_clean = customers_clean.drop(columns=["postal_code"])

droping of postal codes - not too many unique values and not having much predictive capability for us 


In [ ]:
customers_clean["club_member_status"] = customers_clean["club_member_status"].fillna("UNKNOWN")

In [ ]:
customers_clean = customers_clean.drop(columns=["FN_missing", "Active_missing"])

In [ ]:
print(customers_clean.info())

print("\nNull check:")
print(customers_clean.isna().sum())

In [ ]:
print(customers_clean.info())

print("\nNull check:")
print(customers_clean.isna().sum())

In [ ]:
median_age = customers_clean["age"].median()

customers_clean["age"] = customers_clean["age"].fillna(median_age)

In [ ]:
customers_clean["age_bucket"] = pd.cut(
    customers_clean["age"],
    bins=[0, 25, 40, 60, 100],
    labels=[0, 1, 2, 3]
)

In [ ]:
print(customers_clean.info())

print("\nNull check:")
print(customers_clean.isna().sum())

In [ ]:
customers_clean.to_csv(config.CUSTOMERS_CLEAN_PATH, index=False)

print("✅ customers_clean.csv updated")

## TRANSACTIONS DATA PREP

In [ ]:
import sys
from pathlib import Path

# Force project root into path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import config

In [ ]:
import sys
from pathlib import Path

# go to project root
sys.path.append(str(Path().resolve().parent))

import config

In [ ]:
import pandas as pd
import config

In [ ]:
import config
print(config.__file__)

In [ ]:
chunksize = 500_000
start_date = pd.to_datetime("2019-09-22")

In [ ]:
import os

# delete old file if exists
if os.path.exists(config.TRANSACTIONS_SMALL_PATH):
    os.remove(config.TRANSACTIONS_SMALL_PATH)

first_chunk = True

for chunk in pd.read_csv(
    config.TRANSACTIONS_RAW_PATH,
    chunksize=chunksize,
    parse_dates=["t_dat"],
    dtype={
        "customer_id": "str",
        "article_id": "int32",
        "price": "float32",
        "sales_channel_id": "int8"
    }
):
    # 🔹 Filter last 1 year
    chunk = chunk[chunk["t_dat"] >= start_date]

    # 🔹 Keep required columns
    chunk = chunk[["t_dat", "customer_id", "article_id", "price"]]

    # 🔥 MEMORY OPTIMIZATION (IMPORTANT)
    chunk["customer_id"] = chunk["customer_id"].apply(
        lambda x: int(x[-16:], 16)
    ).astype("int64")

    # article_id already int32 (good)
    chunk["price"] = chunk["price"].astype("float32")

    # 🔹 WRITE DIRECTLY TO FILE (NO RAM BUILDUP)
    chunk.to_csv(
        config.TRANSACTIONS_SMALL_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("✅ transactions_small.csv created successfully")

In [ ]:
transactions_small = pd.read_csv(
    config.TRANSACTIONS_SMALL_PATH,
    parse_dates=["t_dat"]
)

print(transactions_small.shape)
transactions_small.head()


In [ ]:
print(transactions_small.isna().sum())
print(transactions_small.dtypes)

In [ ]:
assert "customer_id" in transactions_small.columns
assert "article_id" in transactions_small.columns
assert transactions_small.shape[0] > 0

print("✅ transactions_small verified")

In [ ]:
print(transactions_small["t_dat"].min())
print(transactions_small["t_dat"].max())

In [ ]:
import pandas as pd

# Load your segmented data
df = pd.read_csv(r"C:\Users\koustubh.arole\Desktop\Hm\hm_recommendation_system\data\processed\rfm_segmented.csv")

# Get 5 IDs from Cluster 2 (VIPs)
print("VIP IDs:", df[df['cluster'] == 2]['customer_id'].head(5).tolist())

# Get 5 IDs from Cluster 1 (Lapsed)
print("Lapsed IDs:", df[df['cluster'] == 1]['customer_id'].head(5).tolist())